# 03 · KV cache size from a model config
Level L1 · Part 1 · core · ~25 min · builds on 02

## Goal
By the end of this lesson you can compute the KV cache size per token from a model's `config.json`, check your result against Llama 3.1 8B (128 KiB per token in FP16) and 70B (320 KiB), and say what `num_key_value_heads` means.

## Why it matters
- **Without it:** you learn the cache size by loading the model and watching the GPU run out of memory, or you guess. The Brightside request (1,600 prompt tokens and a 200-token answer) on the 70B model needs about 562 MiB of cache. A guess that is off by 2x changes how many users one GPU can serve.
- **With it:** two or three numbers from `config.json` and one multiplication tell you what every token costs in memory, before you rent a GPU.
- **When you don't need it:** if a hosted provider runs the model and bills you per token, the cache is their memory, not yours. You still pay for the tokens. Lesson 05 turns this math into a concurrency limit.

## The idea
For every token, the model keeps one **key** vector and one **value** vector in every layer, for every **KV head**. So the bytes per token are: 2 (keys and values) × layers × KV heads × head size × bytes per number. Picture a card index with one card per token. Each card holds a key and a value for every layer and every KV head. The number of KV heads is `num_key_value_heads` in the config. It can be smaller than the number of query heads, and that is **grouped-query attention (GQA)**: several query heads share one KV head, so each card is thinner.

## Picture
```mermaid
flowchart LR
  C["config.json: layers, KV heads, head size"] --> F["2 x layers x KV heads x head size x bytes"]
  F --> T["bytes per token"]
  T -->|"x tokens in the request"| R["KV cache of one request"]
```

## Step by step
### Step 1: read the config
The text below holds the fields this lesson uses, copied from the public Llama 3.1 `config.json` files (an ungated mirror, because Meta's repository is gated). Nothing is downloaded and no key is needed. Only numbers are kept, not the whole file.

In [1]:
import json

CONFIGS = {
    "Llama 3.1 8B": """{
  "num_hidden_layers": 32,
  "num_attention_heads": 32,
  "num_key_value_heads": 8,
  "head_dim": 128
}""",
    "Llama 3.1 70B": """{
  "num_hidden_layers": 80,
  "num_attention_heads": 64,
  "num_key_value_heads": 8,
  "hidden_size": 8192
}""",
}
cfg = {name: json.loads(text) for name, text in CONFIGS.items()}
print(CONFIGS["Llama 3.1 70B"])

{
  "num_hidden_layers": 80,
  "num_attention_heads": 64,
  "num_key_value_heads": 8,
  "hidden_size": 8192
}


**What you should see:** the 70B config has no `head_dim`. Its head size is `hidden_size / num_attention_heads` = 8192 / 64 = 128. Real configs differ on this point, so the formula must handle both.

### Step 2: the formula
`bytes_per_value` is 2 for FP16 and BF16, 1 for FP8 and 4 for FP32. The leading `2` is for keys and values.

In [2]:
FP16 = 2  # bytes per number

def kv_bytes_per_token(c, bytes_per_value=FP16):
    """KV cache bytes for one token: keys and values, every layer, every KV head."""
    head_dim = c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]
    return 2 * c["num_hidden_layers"] * c["num_key_value_heads"] * head_dim * bytes_per_value

for name, c in cfg.items():
    per_token = kv_bytes_per_token(c)
    print(f"{name:14} {per_token:>9,} bytes = {per_token / 1024:>4.0f} KiB per token")

Llama 3.1 8B     131,072 bytes =  128 KiB per token
Llama 3.1 70B    327,680 bytes =  320 KiB per token


### Step 3: check against the known numbers

In [3]:
EXPECTED_KIB = {"Llama 3.1 8B": 128, "Llama 3.1 70B": 320}
for name, kib in EXPECTED_KIB.items():
    got = kv_bytes_per_token(cfg[name]) / 1024
    print(f"{name:14} {got:.0f} KiB vs expected {kib} KiB:", "match" if got == kib else "DIFFERENT")
    assert got == kib

Llama 3.1 8B   128 KiB vs expected 128 KiB: match
Llama 3.1 70B  320 KiB vs expected 320 KiB: match


**What you should see:** both lines say `match`. The formula reproduces the numbers from the course outline.

### Step 4: one real request
The Brightside request has a 1,500-token policy, a 100-token question and a 200-token answer (lesson 02). When the answer is finished, the cache holds all 1,800 tokens.

In [4]:
PROMPT, ANSWER = 1_600, 200
tokens = PROMPT + ANSWER
for name, c in cfg.items():
    total = kv_bytes_per_token(c) * tokens
    print(f"{name:14} {tokens:,} tokens -> {total / 2**20:,.1f} MiB of KV cache")

Llama 3.1 8B   1,800 tokens -> 225.0 MiB of KV cache
Llama 3.1 70B  1,800 tokens -> 562.5 MiB of KV cache


**What you should see:** about 225 MiB for the 8B model and 562.5 MiB for the 70B model, for one request. Multiply by the number of requests that run together and you have the memory your server needs. Lesson 05 does that step.

### Step 5: the same models with an FP8 cache
An FP8 cache stores one byte per number instead of two. It halves the cache, but not the weights.

In [5]:
for name, c in cfg.items():
    fp8 = kv_bytes_per_token(c, bytes_per_value=1)
    print(f"{name:14} FP8 cache: {fp8 / 1024:.0f} KiB per token (FP16 was {kv_bytes_per_token(c) / 1024:.0f})")

Llama 3.1 8B   FP8 cache: 64 KiB per token (FP16 was 128)
Llama 3.1 70B  FP8 cache: 160 KiB per token (FP16 was 320)


### Step 6: what `num_key_value_heads` buys
Without GQA, every query head would have its own KV head (multi-head attention, MHA). Compare that with the config.

In [6]:
for name, c in cfg.items():
    q, kv = c["num_attention_heads"], c["num_key_value_heads"]
    mha = kv_bytes_per_token({**c, "num_key_value_heads": q})  # one KV head per query head
    gqa = kv_bytes_per_token(c)
    print(f"{name:14} {q} query heads share {kv} KV heads ({q // kv} per KV head): "
          f"{gqa / 1024:.0f} KiB instead of {mha / 1024:.0f} KiB")

Llama 3.1 8B   32 query heads share 8 KV heads (4 per KV head): 128 KiB instead of 512 KiB
Llama 3.1 70B  64 query heads share 8 KV heads (8 per KV head): 320 KiB instead of 2560 KiB


**What you should see:** the 8B model stores 4 times less than MHA would, and the 70B model 8 times less. Without GQA the 70B cache would be 2,560 KiB (2.5 MiB) per token. GQA is not free: sharing KV heads can cost some quality, which you measure rather than assume. Lesson 04 compares MHA, MQA, GQA and MLA.

## Under the hood
The formula counts what the attention layers store. In one layer, each token gets a key vector and a value vector per KV head, and each vector has `head_dim` numbers.

For Llama 3.1 8B, one layer stores 2 × 8 × 128 = 2,048 numbers per token. Over 32 layers that is 65,536 numbers, or 131,072 bytes at 2 bytes each: 128 KiB.

| Factor | Llama 3.1 8B | Llama 3.1 70B | What it counts |
|---|---|---|---|
| 2 | 2 | 2 | keys and values |
| layers | 32 | 80 | one cache slice per layer |
| KV heads | 8 | 8 | vectors per slice (GQA) |
| head size | 128 | 128 | numbers per vector |
| bytes | 2 | 2 | FP16 |
| **per token** | **128 KiB** | **320 KiB** | product, divided by 1,024 |

## At work
- **Databricks** gives about 20 GiB of KV cache for Llama-2-70B at batch 64 with 1,024 tokens each in FP16. The same formula gives that number, so use it to check your own config before a load test. → lesson 05.
- **Cloudflare (May 2026)** reports more than 56 GiB of KV room on 2 H200 GPUs for Llama 4 Scout, which supports over 1.2 million tokens. Long context is paid in cache memory on every request that uses it. → lesson 05.
- **Interview prep sources** list "KV cache size and max concurrency for model X on GPU Y" as a common system-design question (medium-low confidence: a synthesis, not a survey). → lesson 05.

Sources: Databricks, "LLM Inference Performance Engineering: Best Practices": https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices · Cloudflare, "High-performance LLMs" (May 2026): https://blog.cloudflare.com/high-performance-llms/

## Common mistakes
**1. Reading `head_dim` directly.** Some configs (the 70B here) leave it out, and the lookup fails.

In [7]:
print(cfg["Llama 3.1 70B"]["head_dim"])  # KeyError: this config has no head_dim

KeyError: 'head_dim'

The fix is the `.get` with the `hidden_size / num_attention_heads` fallback in Step 2.

**2. Using the query head count.** The cache stores KV heads, not query heads. The wrong line below treats 64 query heads as KV heads.

In [8]:
wrong = 2 * 80 * 64 * 128 * 2   # 64 query heads: wrong
right = 2 * 80 * 8 * 128 * 2    # 8 KV heads: right
print(f"wrong: {wrong / 1024:.0f} KiB per token, right: {right / 1024:.0f} KiB per token")

wrong: 2560 KiB per token, right: 320 KiB per token


**3. Mixing up MB and MiB.** 131,072 bytes is 128 KiB, but it is 0.131 MB in decimal units. Blogs use both, so check which one a number uses before you compare.

In [9]:
print(f"131,072 bytes = {131_072 / 1024:.0f} KiB = {131_072 / 1_000_000:.3f} MB")

131,072 bytes = 128 KiB = 0.131 MB


**4. Forgetting the factor 2 or the byte size.** Dropping the 2 for keys and values halves the answer. Using FP32 (4 bytes) doubles it.

## Try it
Your team moves the 70B model to an FP8 cache. Write code that prints the KV cache per token in KiB, and the cache for a 4,000-token chat in MiB. Use `kv_bytes_per_token`.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
per_token = kv_bytes_per_token(cfg["Llama 3.1 70B"], bytes_per_value=1)
print(f"{per_token / 1024:.0f} KiB per token, {4_000 * per_token / 2**20:.0f} MiB for a 4,000-token chat")

160 KiB per token, 625 MiB for a 4,000-token chat


## Recap
- KV cache bytes per token = 2 × layers × KV heads × head size × bytes per number.
- The config gives layers and KV heads. The head size is `head_dim`, or `hidden_size / num_attention_heads` when `head_dim` is missing.
- `num_key_value_heads` below `num_attention_heads` is GQA. It cuts the cache by the group size (4x for 8B, 8x for 70B).
- Llama 3.1 8B needs 128 KiB per token in FP16, and 70B needs 320 KiB. FP8 halves both.

**Interview:** "How much KV cache does a 70B model need per token?" Strong answer: read layers, KV heads and head size from the config, then compute 2 × 80 × 8 × 128 × 2 bytes = 320 KiB in FP16. It is GQA, so the KV heads are 8, not the 64 query heads.